# Data Filtering through Image Quality Assessment

This notebook demonstrates how to filter data using image quality metrics.
Per-image metrics and max intensity projection metrics are computed and
visualised across the lowest, mid, and highest quality bins.

In [ ]:
import glob
import os
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from skimage import io

from src.dataset_analysis.image_quality_metrics import (
    get_image_quality_metrics,
    get_percentile_images,
    max_intensity_projection_metrics,
)

In [ ]:
# CONFIG
FOLDER = "../data/Plate 2426/BF images"
METRICS = ["laplacian_variance", "fft_sharpness", "shannon_entropy"]

In [ ]:
images = glob.glob(os.path.join(FOLDER, "*.tif"))
images = [img for img in images if "w1" in img]

metrics = get_image_quality_metrics(images)

In [ ]:
metrics_df = pd.DataFrame.from_dict(metrics, orient="index")
metrics_df.head()

In [ ]:
for i in range(5):
    fname = list(metrics.keys())[i]
    print(f"{fname}: {metrics[fname]}")

In [ ]:
results = {metric: get_percentile_images(metrics, metric) for metric in METRICS}

for metric_name, groups in results.items():
    print(f"\nMetric: {metric_name}")
    print("Highest quality images:", groups["highest"])
    print("Mid quality images:    ", groups["mid"])
    print("Lowest quality images: ", groups["lowest"])

In [ ]:
def plot_quality_grid(group, title="", image_map=None):
    """3x3 grid: top row = highest, middle = mid, bottom = lowest quality."""
    all_imgs = group["highest"] + group["mid"] + group["lowest"]
    fig, axes = plt.subplots(3, 3, figsize=(12, 12))
    for ax, img_id in zip(axes.flat, all_imgs):
        img = image_map[img_id] if image_map is not None else io.imread(img_id)
        if img.ndim == 3:
            img = img.mean(axis=-1)
        ax.imshow(img, cmap="gray")
        ax.set_title(os.path.basename(img_id))
        ax.axis("off")
    if title:
        fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_quality_grid(results["fft_sharpness"], title="FFT Sharpness")

In [ ]:
plot_quality_grid(results["laplacian_variance"], title="Laplacian Variance")

## Max Intensity Projections

Group z-slices by well/site, compute the max intensity projection across z, then re-score quality.

In [ ]:
base_id_groups = defaultdict(list)
for path in images:
    basename = os.path.basename(path)
    base_id = basename[: basename.rfind("_z")]
    base_id_groups[base_id].append(path)

print(f"Total unique base IDs: {len(base_id_groups)}")

In [ ]:
projections, projection_metrics = max_intensity_projection_metrics(base_id_groups)

In [ ]:
results_projections = {
    metric: get_percentile_images(projection_metrics, metric) for metric in METRICS
}

for metric_name, groups in results_projections.items():
    print(f"\nMetric: {metric_name}")
    print("Highest quality images:", groups["highest"])
    print("Mid quality images:    ", groups["mid"])
    print("Lowest quality images: ", groups["lowest"])

In [ ]:
plot_quality_grid(
    results_projections["laplacian_variance"],
    title="Laplacian Variance (Max Intensity Projection)",
    image_map=projections,
)

## Saving outputs

To execute this notebook and write a rendered copy with all outputs to `notebooks/output/`, run:

```bash
jupyter nbconvert --to notebook --execute notebooks/00_data_filter.ipynb \
    --output-dir notebooks/output/
```

Outputs are excluded from version control (see `.gitignore`).  
The source notebook is kept output-free by the `nbstripout` pre-commit hook.